# AquaVision - Segmenter **augmented** runs on Kaggle (A / B / C / D)

Same data, converter, split (`--size_strata 2`, seed 42), model (`yolo11s-seg.pt`), `imgsz 640`, `batch 16`,
max 100 epochs + `patience 25`, and the same evaluation script (`eval_seg.py`) as **baseline v3**
(`train_segmenter_kaggle.ipynb`). Only the TRAIN data / augmentation changes. One run per Commit:
set `RUN` in the next cell, then **Save Version -> Save & Run All (Commit)**.

| RUN | Train data | Online aug | Answers (brief) | GPU (est.) |
|---|---|---|---|---|
| **A** | originals only | geometric + mosaic/mixup (`GEOMETRIC`) | P1 overfit, P2 fish size | ~1-2.5 h |
| **B** | originals + 2 photometric `__aug` copies | same as A | + P4 turbid/domain | ~2.5-6 h |
| **C** | B + 10 % `__ovl` overlays + 10 % `__comp` DeepFish->Brackish composites; two-phase (last 15 ep: no mosaic/overlays/composites, SGD lr 1e-3) | same as A | + P5 clutter, P3 real-mask fish | ~3-6.5 h |
| **D** *(optional)* | C, but Brackish boxes -> GrabCut **pseudo-masks** (`--pseudo_masks`) | same as A | P3 masks for length | C + ~40 min prep |

**D caveat:** pseudo-masks change the Brackish TEST labels too, so D's *mask* mAP is not comparable with the baseline
(box mAP still is). Judge D by box mAP + by looking at the predicted masks.

**Kaggle setup:** GPU T4 x2, Internet On. Attach **`aquavision-scripts`** (needs the updated `prepare_data.py`,
`augment_offline.py`, `turbid_split.py`, `train_aug_config.py` and the new `eval_seg.py`) and **`aquavision-brackish`**.

## 0 - Choose the run

In [ ]:
RUN = "A"            # <- "A", "B", "C" or "D"  (one per Commit)
assert RUN in ("A", "B", "C", "D")
PLAN = {"A": "originals + online geometric", "B": "+ photometric copies", "C": "+ overlays/composites, two-phase",
        "D": "C + pseudo-masks"}
print("RUN", RUN, "->", PLAN[RUN])

## 1 - GPU + install

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics huggingface_hub
import torch
assert torch.cuda.is_available(), "No CUDA torch - set Accelerator=GPU and do not upgrade torch"
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0))

## 2 - Scripts + config

In [ ]:
import os, shutil, glob
def find_input(name):                                # /kaggle/input/<name> or /kaggle/input/datasets/<user>/<name>
    hits = glob.glob(f"/kaggle/input/{name}") + glob.glob(f"/kaggle/input/*/*/{name}")
    assert hits, f"{name} not attached - Add Input -> {name} (see !ls /kaggle/input)"
    return hits[0]
SCRIPTS     = find_input("aquavision-scripts")
RF_BRACKISH = find_input("aquavision-brackish")
print("SCRIPTS =", SCRIPTS, "| contains:", sorted(os.listdir(SCRIPTS)))
REPO = "/kaggle/working/aquavision"; os.makedirs(REPO, exist_ok=True)
NEED = {"prepare_data.py": "--size_strata", "augment_offline.py": "one masked dataset is enough",
        "underwater_aug.py": "build_underwater_transforms", "train_aug_config.py": 'optimizer="SGD"',
        "turbid_split.py": "lambda _, k=k, v=v", "eval_seg.py": "recall_by_size"}
for f, marker in NEED.items():
    assert os.path.exists(f"{SCRIPTS}/{f}"), f"{f} missing from aquavision-scripts - dataset -> New Version -> add it"
    shutil.copy(f"{SCRIPTS}/{f}", f"{REPO}/{f}")
    assert marker in open(f"{REPO}/{f}").read(), f"{f} in aquavision-scripts is an OLD version - upload the current one"
os.chdir(REPO); print("cwd:", os.getcwd(), "| scripts OK")

DEEPFISH = "/kaggle/working/data/deepfish"
BRACKISH = "/kaggle/working/data/brackish_frames"
WORK = "/kaggle/working/work"; DET = f"{WORK}/det_aug"
RUNS = "/kaggle/working/runs"
EPOCHS_DET, PATIENCE, LATE = 100, 25, 15            # same budget as baseline v3; LATE = phase-2 epochs (C/D)

## 3 - Get DeepFish (Hugging Face, ~7 GiB)

In [ ]:
import os, glob, tarfile
os.makedirs("/kaggle/working/data", exist_ok=True)
from huggingface_hub import snapshot_download
snapshot_download("Alzayats/DeepFish", repo_type="dataset", local_dir="/kaggle/working/data/deepfish")
for t in glob.glob("/kaggle/working/data/deepfish/*.tar"):
    tarfile.open(t).extractall("/kaggle/working/data/deepfish"); os.remove(t)
print("---- DeepFish tree (find the Segmentation folder: images + masks/) ----")
!ls -R /kaggle/working/data/deepfish | head -50
# >>> set DEEPFISH to the Segmentation subfolder shown above, e.g.:
DEEPFISH = "/kaggle/working/data/deepfish/DeepFish/Segmentation"
print("DEEPFISH =", DEEPFISH)

## 4 - Convert Brackish (Roboflow) -> `<clip>_frame_<NNNN>` fish frames (identical to the baseline)

In [ ]:
# Convert the attached Roboflow Brackish dataset -> single-class 'fish' frames for prepare_data.
# FAST: symlinks (no file copy) + keeps only fish-containing frames + live progress.
# Files are renamed <clip>_frame_<NNNN> so prepare_data's group_key() splits by video clip (no leakage).
import re, os, shutil, yaml
from collections import Counter
from pathlib import Path
RF = Path(RF_BRACKISH)
BR = Path(BRACKISH)
shutil.rmtree(BR, ignore_errors=True)                 # clear any partial output
BR.mkdir(parents=True, exist_ok=True)
dy = next(RF.rglob("data.yaml"), None)
assert dy is not None, f"no data.yaml under {RF} - is the Brackish dataset attached and named right? (see /kaggle/input)"
names = yaml.safe_load(dy.read_text())["names"]
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
norm = lambda n: re.sub(r"[\s\-]+", "_", str(n).strip().lower())
fish_idx = {i for i, n in enumerate(names) if norm(n) in {"fish", "small_fish", "smallfish"}}
print("classes:", names, "-> fish indices:", sorted(fish_idx), flush=True)

RF_NAME = re.compile(r"^(?P<clip>.+)-(?P<frame>\d+)_(?:jpg|jpeg|png)\.rf\.[0-9a-f]+$", re.I)
imgs = [p for e in ("*.jpg", "*.jpeg", "*.png") for p in RF.rglob(e)]
print(f"{len(imgs)} Brackish images found; converting (symlink, fish-only)...", flush=True)
kept = boxes = dup = 0
clips = Counter()
for k, im in enumerate(imgs, 1):
    m = RF_NAME.match(im.stem)
    assert m, f"unexpected Roboflow file name (cannot find clip/frame): {im.name}"
    lab = im.parent.parent / "labels" / f"{im.stem}.txt"
    if not lab.exists(): lab = im.with_suffix(".txt")
    bx = []
    if lab.exists():
        for line in lab.read_text().splitlines():
            v = line.split()
            if len(v) >= 5 and int(float(v[0])) in fish_idx:
                bx.append("1 " + " ".join(v[1:5]))
    if not bx:                                         # skip frames with no fish (fast + fish-relevant)
        continue
    stem = f"{m['clip']}_frame_{int(m['frame']):04d}"
    dst = BR / f"{stem}{im.suffix.lower()}"
    if dst.exists():                                   # same clip frame exported twice -> keep one
        dup += 1
        continue
    os.symlink(im, dst)                                # instant, no copy
    (BR / f"{stem}.txt").write_text("\n".join(bx))
    kept += 1; boxes += len(bx); clips[m["clip"]] += 1
    if k % 2000 == 0:
        print(f"  {k}/{len(imgs)} scanned, {kept} fish frames kept", flush=True)
print(f"DONE: {kept} Brackish fish frames ({boxes} boxes) from {len(clips)} clips -> {BR}"
      f"  ({dup} duplicate frames skipped)", flush=True)
assert kept, "no fish frames found - check the Brackish dataset is attached"
BRACKISH = str(BR)

## 5 - Prepare detector data - IDENTICAL split to baseline v3 (`--size_strata 2`, seed 42)
Run D adds `--pseudo_masks` (GrabCut outlines inside every Brackish box, ~20-40 min on CPU; split unchanged).

In [ ]:
PSEUDO = "--pseudo_masks" if RUN == "D" else ""
!python prepare_data.py detector --deepfish "{DEEPFISH}" --brackish "{BRACKISH}" --out "{DET}" --bg_ratio 1.0 --size_strata 2 --seed 42 {PSEUDO}

import csv, sys, re
from collections import defaultdict
from pathlib import Path
sys.path.insert(0, REPO)
from prepare_data import group_key
seen = defaultdict(set)
for r in csv.DictReader(open(f"{DET}/manifest.csv")):
    seen[(r["dataset"], group_key(r["file"].split("__", 1)[1].rsplit(".", 1)[0]))].add(r["split"])
leaks = [k for k, v in seen.items() if len(v) > 1]
assert not leaks, f"LEAKAGE: {len(leaks)} groups in more than one split, e.g. {leaks[:3]}"
n_test = sum(1 for r in csv.DictReader(open(f"{DET}/manifest.csv")) if r["split"] == "test")
print(f"OK: no clip in more than one split | test images: {n_test}  (must equal baseline v3's test count)")

## 6 - Offline augmentation (TRAIN only) + the yaml for this run
* A: none - `data_orig.yaml` (originals only)
* B: 2 photometric `__aug` copies per train image (+ DeepFish<->Brackish colour transfer) -> `data_full.yaml`
* C/D: B + 10 % `__ovl` overlays + 10 % `__comp` composites -> `data_full.yaml` (phase 1), `data_late.yaml` (phase 2)

In [ ]:
det = Path(DET)
if RUN != "A":
    frac = 0.1 if RUN in ("C", "D") else 0.0
    !python augment_offline.py --det "{DET}" --copies 2 --domain_mix --overlay_frac {frac} --composite_frac {frac} --max_side 960 --workers 0 --seed 0

# originals-only list (Run A trains on it; also a record of the real train images)
TAGS = ("__aug", "__ovl", "__comp")
orig = [p for p in sorted((det / "images" / "train").glob("*.jpg")) if not any(t in p.stem for t in TAGS)]
(det / "lists").mkdir(exist_ok=True)
(det / "lists" / "train_orig.txt").write_text("\n".join(str(p.resolve()) for p in orig) + "\n")
(det / "data_orig.yaml").write_text(re.sub(r"(?m)^train:.*$",
    lambda _: f"train: {(det / 'lists' / 'train_orig.txt').resolve()}", (det / "data.yaml").read_text()))

for s in ("val", "test"):                              # copies must never leave train/
    bad = [p.name for p in (det / "images" / s).glob("*.jpg") if any(t in p.stem for t in TAGS)]
    assert not bad, f"augmented copies in {s}: {bad[:3]}"
DATA = str(det / ("data_orig.yaml" if RUN == "A" else "data_full.yaml"))
lst =re.search(r"(?m)^train:\s*(.*)$", open(DATA).read()).group(1).strip()
print(f"RUN {RUN}: training on {DATA}\n  train images: {sum(1 for _ in open(lst)) if lst.endswith('.txt') else '?'}"
      f" (originals {len(orig)})")

## 7 - Look at a few augmented training images (labels drawn; green = real-mask polygon, red = box)

In [ ]:
import cv2, numpy as np
from IPython.display import Image, display
picks = []
for t in ("__comp0", "__ovl0", "__aug0", "__aug1"):
    c = sorted((det / "images" / "train").glob(f"brackish__*{t}.jpg"))
    if c: picks.append(c[len(c) // 2])
tiles = []
for p in picks:
    im = cv2.imread(str(p)); H, W = im.shape[:2]
    for l in (det / "labels" / "train" / f"{p.stem}.txt").read_text().splitlines():
        v = np.array(l.split()[1:], np.float32).reshape(-1, 2) * [W, H]
        cv2.polylines(im, [v.astype(np.int32)], True, (0, 255, 0) if len(v) > 4 else (0, 0, 255), 2)
    cv2.putText(im, p.stem.rsplit("__", 1)[1], (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    tiles.append(cv2.resize(im, (640, 360)))
if tiles:
    while len(tiles) % 2: tiles.append(np.zeros_like(tiles[0]))
    grid = np.vstack([np.hstack(tiles[i:i + 2]) for i in range(0, len(tiles), 2)])
    cv2.imwrite(f"{WORK}/aug_preview.jpg", grid); display(Image(filename=f"{WORK}/aug_preview.jpg"))
else:
    print("Run A: no offline copies (online augmentation only)")

## 8 - Train
A/B: one run, `GEOMETRIC` online aug from `train_aug_config.py`, mosaic/mixup off for the last 15 epochs (`close_mosaic`).
C/D: `train_aug_config.train_detector` - phase 1 on `data_full.yaml` (patience 25), then 15 epochs on
`data_late.yaml` with mosaic/mixup off, SGD lr0 1e-3, no warm-up.

In [ ]:
from ultralytics import YOLO
import train_aug_config as tac
print("online aug:", tac.GEOMETRIC)
if RUN in ("A", "B"):
    m = YOLO("yolo11s-seg.pt")                     # COCO-pretrained, same start as the baseline
    m.train(data=DATA, epochs=EPOCHS_DET, patience=PATIENCE, close_mosaic=LATE, imgsz=640, batch=16, device=0,
            workers=2, project=RUNS, name=f"seg_aug_{RUN}", exist_ok=True, save_period=10, **tac.GEOMETRIC)
    RUN_DIR = str(m.trainer.save_dir)
else:
    m = tac.train_detector(det_dir=DET, weights="yolo11s-seg.pt", epochs=EPOCHS_DET, late_epochs=LATE,
                           imgsz=640, batch=16, patience=PATIENCE, project=RUNS, name=f"seg_aug_{RUN}")
    RUN_DIR = str(m.trainer.save_dir)              # phase-2 folder holds the final best.pt
    p1 = f"{RUNS}/seg_aug_{RUN}_p1"
    for f in ("results.csv", "results.png", "args.yaml"):   # keep phase-1 curves next to phase 2
        if os.path.exists(f"{p1}/{f}"): shutil.copy(f"{p1}/{f}", f"{RUN_DIR}/phase1_{f}")
print("run folder:", RUN_DIR)

## 9 - Test with `eval_seg.py` (same script + same test frames as baseline v3)

In [ ]:
!python eval_seg.py --weights "{RUN_DIR}/weights/best.pt" --det "{DET}" --out "{RUN_DIR}" --project "{RUNS}"

## 10 - Files to download
Everything is in the run folder printed below (`runs/seg_aug_A`, `seg_aug_B`, or `seg_aug_C_p2` / `seg_aug_D_p2`).
The raw downloads and the augmented image cache are deleted afterwards so the Output tab stays small.

In [ ]:
for f in ("manifest.csv", "data.yaml", "raw_counts.json", "turbidity.csv"):
    if os.path.exists(f"{DET}/{f}"): shutil.copy(f"{DET}/{f}", RUN_DIR)
if os.path.exists(f"{WORK}/aug_preview.jpg"): shutil.copy(f"{WORK}/aug_preview.jpg", RUN_DIR)
print("RUN", RUN, "->", RUN_DIR)
for f in ["weights/best.pt", "results.csv", "args.yaml", "test_metrics.json", "manifest.csv", "turbidity.csv",
          "results.png", "test_plots/MaskPR_curve.png", "test_plots/confusion_matrix.png"] + \
         (["phase1_results.csv"] if RUN in ("C", "D") else []) + ([] if RUN == "A" else ["aug_preview.jpg"]):
    p = f"{RUN_DIR}/{f}"
    print(f"OK      {p}  ({os.path.getsize(p) / 1e6:.1f} MB)" if os.path.exists(p) else f"MISSING {p}")
shutil.rmtree("/kaggle/working/data", ignore_errors=True)   # raw DeepFish + Brackish symlinks
shutil.rmtree(WORK, ignore_errors=True)                       # prepared + augmented images (GBs)
print("cleaned /kaggle/working/data and", WORK)